## Jesuit communities in China ("Chrétientés") by Dehergne

Based on Dehergne, J. (1973). Répertoire des Jésuites de Chine de 1552 à 1800. Institutum historicum ; Letouzey & Ané. https://archive.org/details/bhsi37

Complemented with Dehergne, J. (1957). Les Chrétientés de Chine de la Période Ming (1581–1650). Monumenta Serica, 16(1–2), 1–136. https://doi.org/10.1080/02549948.1957.11730960 where more detailed information is provided.

The Répertoire includes two lists of Jesuits residences in China. One reports to 1644 and the second to 1701.

- [Planche: Carte des Chrétientés Chinoises de la fin des Ming (1644)](https://archive.org/details/bhsi37/page/398/mode/1up) and [map](https://archive.org/details/bhsi37/page/n390/mode/1up) Kleiotranscription at [sources/dehergne-locations-1644.cli](../sources/dehergne-locations-1644.cli)
- [VII. Carte des résidences de Chine en l'année 1701](https://archive.org/details/bhsi37/page/n400/mode/1up) and [map](https://archive.org/details/bhsi37/page/n392/mode/1up) kleio transcription at [sources/dehergne-locations-1701.cli](../sources/dehergne-locations-1701.cli).

### Overview
This notebook produces tables with the names of communties, their wkidata code, and comments related to
modern chinese spelling of the place name. 

The notebook also uses records related to place names in individual missionary entries in the "Repertoire"
to list all the entities present in a community at a given point in time.

### Output

**Excel files produced in this notebook (inferences directory):**
- [residences-1644.xlsx](../inferences/residences-1644.xlsx)
- [residences-1701.xlsx](../inferences/residences-1701.xlsx)
- [residences-1644-1701.xlsx](../inferences/residences-1644-1701.xlsx)
- [residences-1644-1701-no-wikidata.xlsx](../inferences/residences-1644-1701-no-wikidata.xlsx)

**Map produced:**
- [residences_1644_map.html](../inferences/residences_1644_map.html)
- [residences_1701_map.html](../inferences/residences_1701_map.html)


### Setup

In [ ]:
import timelink
import timelink.notebooks

print("Timelink version:", timelink.__version__)

tlnb = timelink.notebooks.TimelinkNotebook(kleio_image="kleio-server", kleio_version="12.9.592")
# tlnb.print_info()

## List of residences in 1644

Loop through the list of residences in 1644 in file `dehergne-locations-1644.cli` and 
build a dataframe with the administrative structure and the wikidata of the locations. 


In [ ]:
import pandas as pd
from sqlalchemy import select
from dehergne_util import geo_entity_wikidata_id

geo1, geo2, geo3 = tlnb.db.get_model(['geo1','geo2','geo3'])
stmt = select(geo1).where(geo1.inside == 'deh-chre-1644').order_by(geo1.the_line)

place_list = []

with tlnb.db.session() as session:
    result = session.execute(stmt).fetchall()
    for province, in result:
        comment, wikidata = geo_entity_wikidata_id(province, if_missing='No wikidata')
        # print(province.id, province.name, wikidata, comment)
        place_list.append({'province': province.name,
                           'id': province.id,
                           'level': 'province',
                           'name': province.name,
                           'name_original': province.with_extra_info().name,
                           'province_wikidata_id': wikidata,
                           'wikidata_id': wikidata,
                           'the_line': province.the_line,
                           'comment': comment})
        fous = session.execute(select(geo2).where(geo2.inside == province.id)).fetchall()
        province_wikdata = wikidata
        for fou, in fous:
            the_type = fou.the_type
            comment, wikidata = geo_entity_wikidata_id(fou, if_missing='No wikidata')
            # print(' ', fou.name,  wikidata, comment)
            place_list.append({'province': province.name,
                               'province_id': province.id,
                               'province_wikidata_id': province_wikdata,
                               'id': fou.id,
                               'level': 'fou',
                               'fou':fou.name,
                               'name': fou.name,
                               'name_original': fou.with_extra_info().name,
                               'fou_wikidata_id': wikidata,
                               'wikidata_id': wikidata,
                               'the_line': fou.the_line,
                               'comment': comment})
            geo3s = session.execute(select(geo3).where(geo3.inside == fou.id)).fetchall()
            fou_wikidata = wikidata
            for tcheou_hien, in geo3s:
                comment, wikidata = geo_entity_wikidata_id(tcheou_hien, if_missing='No wikidata')
                # print('   ', tcheou_hien.name, wikidata,comment,  )
                place_list.append({'province': province.name,
                                    'province_id': province.id,
                                    'province_wikidata_id': province_wikdata,
                                    'fou_id': fou.id,
                                    'fou_wikidata_id': fou_wikidata,
                                    'id': tcheou_hien.id,
                                    'level': 'tcheou-hien',
                                    'fou':fou.name,
                                    'name': tcheou_hien.name,
                                    'name_original': tcheou_hien.with_extra_info().name,
                                    'wikidata_id': wikidata,
                                    'the_line': tcheou_hien.the_line,
                                    'comment': comment})


#### Create a dataframe with the locations in 1644



In [ ]:
pd.set_option('display.max_rows', 300)
# create a dataframe from the list
places_1644_df = pd.DataFrame(place_list)
places_1644_df['year'] = 1644
cols=['year','id','level','province',  'fou','name', 'name_original', 'province_wikidata_id', 'fou_wikidata_id','wikidata_id', 'comment','the_line']
places_1644_df.info()
# 'the_line' is the order of the places in the list
places_1644_df[cols].sort_values(by=['the_line']).head()

### Export the locations in 1644 

Export list of residences in 1644 to a file and list of wikidata
ids so they can be used to fetch coordinates and other information.

Ensure that notebook `wikidata-linked-data.ipynb` is run after the next cell to update cache of linked data.


In [ ]:
places_1644_df.to_excel('../inferences/residences-1644.xlsx', index=False)
# save to wikidata ids directory
places_1644_df['wikidata_id'].to_csv('../inferences/wikidata-references/residences-1644.csv', index=False)

Also export a text (kleio) file:

In [ ]:
with tlnb.db.session() as session:
    residences_1644 = tlnb.db.get_entity('deh-chre-1644',session=session)
    print(residences_1644.to_kleio())

### Link with wikidata data

Requires that `wikidata-linked-data.ipynb` has been run first to generate the `locations_wikidata_info` file used here.


In [ ]:
# load wikidata data
from dehergne_util import locations_wikidata_info_file

# load the locations wikidata info from xlsx file
print("Loading locations wikidata info from", locations_wikidata_info_file)
locations_wikidata_info = pd.read_excel(locations_wikidata_info_file, dtype=str)
locations_wikidata_info.set_index('wikidata_id', inplace=True)

# merge the two dataframes
merged_df = pd.merge(places_1644_df, locations_wikidata_info, on='wikidata_id', how='left')
cols = ['level', 'province', 'fou', 'name', 'wikidata_id', 'coordinates','latitude','longitude','comment', 'province_wikidata_id', 'fou_wikidata_id']
merged_df[cols].sort_values(by=['province', 'fou', 'name']).head(10)

Missing coordinates

In [ ]:
# list rows where coordinates are not available
missing_coords = merged_df[merged_df['coordinates'].isna()]
print(f"Rows with missing coordinates ({len(missing_coords)}):")
missing_coords[cols].sort_values(by=['province', 'fou', 'name'])


### Scan for coordinates

If there is no corresponding wikidata entry, or the wikidata entry does not have coordinates, but a geo location was obtained it is introduced in the comments of the transcription and can be retrived here.

In [ ]:
import re
from dehergne_util import extract_coordinates

# set the coordinates for the places that have coordinates in the comment column
# interate rows with iterrows
for index, row in merged_df[merged_df['coordinates'].isna()].iterrows():
    # print()
    # print("Looking for coordinates of:", row['id'], row['name'])
    # if row['comment'] is not NaN and contains "coordinates", "latitude", or "longitude"
    if not re.search(r'coordinates:|latitude:|longitude:', row['comment'], flags=re.IGNORECASE):
        # print("No coordinates found in comment for row:", index, row['name'])
        continue
    coords = extract_coordinates(row['comment'])
    if coords:
        print()
        print(row['comment'])
        print("Found coordinates:", coords)
        merged_df.at[index, 'coordinates'] = coords
        merged_df.at[index, 'latitude'] = coords[0]
        merged_df.at[index, 'longitude'] = coords[1]


### List places with no coordinates

These are the ones that where not geo referenced so far.

In [ ]:
# List rows in merged_df where the 'coordinates' column is NaN
missing_coords_df = merged_df[merged_df['coordinates'].isna()]
missing_coords_df

### Map the structure of residences in 1644

In [ ]:
%pip install plotly

In [ ]:
from textwrap import wrap
from IPython.display import HTML, display
import plotly.graph_objects as go

# generate a dictionary for quick lookup with id as key
locations_wikidata_dict = {row['id']: row for index, row in merged_df.iterrows()}

# Define marker styles for each level
level_styles = {
    'province': {'color': 'red', 'size': 12},
    'fou': {'color': 'blue', 'size': 8},
    'tcheou-hien': {'color': 'green', 'size': 5}
}

fig = go.Figure()


for level, style in level_styles.items():
    fig.add_trace(go.Scattermap(
        lon=[None],
        lat=[None],
        mode='markers',
        marker=dict(color=style['color'], size=style['size']),
        name=level
    ))
# Add lines for 'fou' and 'tcheou_hien'
for _, row in merged_df.iterrows():
    level = row.get('level')
    if level == 'fou':
        id_origin = row.get('province_id', None)
        id_destination = row.get('id', None)
        lat_origin = locations_wikidata_dict.get(id_origin, {}).get('latitude', None)
        lon_origin = locations_wikidata_dict.get(id_origin, {}).get('longitude', None)
        lat_destination = locations_wikidata_dict.get(id_destination, {}).get('latitude', None)
        lon_destination = locations_wikidata_dict.get(id_destination, {}).get('longitude', None)
        if lat_origin and lon_origin and lat_destination and lon_destination:
            fig.add_trace(go.Scattermap(
                lon=[float(lon_origin), float(lon_destination)],
                lat=[float(lat_origin), float(lat_destination)],
                mode='lines',
                line=dict(width=1, color='red'),
                showlegend=False
            ))
    elif level == 'tcheou-hien':
        id_origin = row.get('fou_id', None)
        id_destination = row.get('id', None)
        lat_origin = locations_wikidata_dict.get(id_origin, {}).get('latitude', None)
        lon_origin = locations_wikidata_dict.get(id_origin, {}).get('longitude', None)
        lat_destination = locations_wikidata_dict.get(id_destination, {}).get('latitude', None)
        lon_destination = locations_wikidata_dict.get(id_destination, {}).get('longitude', None)
        if id_destination == 'Q7420445' or row['name'] == "Camarri": # special case for Taparri
            print(f"Debugging: {id_origin} -> {id_destination}")
            pass
        if lat_origin and lon_origin and lat_destination and lon_destination:
            fig.add_trace(go.Scattermap(
                lon=[float(lon_origin), float(lon_destination)],
                lat=[float(lat_origin), float(lat_destination)],
                mode='lines',
                line=dict(width=1, color='blue'),
                showlegend=False
            ))
# Add Markers for each location
for index, row in merged_df.iterrows():
    lat = row.get('latitude')
    lon = row.get('longitude')
    level = row.get('level')
    name = row.get('name')
    english_description = row.get('english_description', '')
    chinese_description = row.get('chinese_description', '')
    wikidata_id = row.get('wikidata_id', '')
    coordinates = row.get('coordinates', '')
    if name == "Camarri":
        print(f"Debugging: {name} with wikidata_id {wikidata_id}")

    if wikidata_id != 'No wikidata':
        english_label = row.get('english_label', '')
        chinese_label = row.get('chinese_label', '')
        portuguese_label = row.get('portuguese_label', '')
        english_description = row.get('english_description', '')
        chinese_description = row.get('chinese_description', '')
        portuguese_description = row.get('portuguese_description', '')
        wikidata_label = (
                         f"en:{english_label} ({english_description})<br>"
                         f"zh:{chinese_label} ({chinese_description})<br>"
                         f"pt:{portuguese_label} ({portuguese_description})"
        )
    else:
        wikidata_label = "No wikidata"
        english_label = ''
        chinese_label = ''
        portuguese_label = ''
        english_description = ''
        chinese_description = ''
        portuguese_description = ''

    name_original = row.get('name_original', '')
    comment = "<br>".join(wrap(name_original, width=40))
    hover_text = f"name: {name}<br>original: {comment}<br><br>wikidata ({wikidata_id})<br>{wikidata_label} <br>coordinates: {coordinates}"


    if level == 'tcheou-hien':
        pass
    if pd.notnull(lat) and pd.notnull(lon) and level in level_styles:
        style = level_styles[level]
        fig.add_trace(go.Scattermap(
            lon=[float(lon)],
            lat=[float(lat)],
            mode='markers+text',
            marker=dict(color=style['color'], size=style['size']),
            text=[name],
            textposition='top center',
            hovertext=[hover_text],
            hoverinfo='text',
            hoverlabel=dict(bgcolor='white', font_size=12),
            showlegend=False
        ))

# fig.update_layout(mapbox_style="open-street-map")
# fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.update_traces(textfont=dict(size=6))
fig.update_layout({
    "map": {"center": {"lat": 28.0,
                       "lon": 120.0},
            "zoom": 4,
            "style": "carto-positron",
            }
    },
    title= "Residences 1644",
    annotations=[dict(
        x=0,
        y=-0.1,
        xref="paper",
        yref="paper",
        text="<a href='https://archive.org/details/bhsi37/page/n390/mode/1up'>Source</a>",
        showarrow=False,
        font=dict(size=10)
    )],
    autosize=True,
    )

display(HTML(fig.to_html(include_plotlyjs=True, config={"responsive": True})))
fig.write_html("../inferences/residences_1644_map.html",
               config={"responsive": True},
               include_plotlyjs=True,
               auto_open=True)


## List of residences in 1701

Loop through the list of residences in 1701 in file `dehergne-locations-1701.cli` and create a dataframe with the administrative structure and the wikidata of the locations.

In [ ]:
import pandas as pd
from sqlalchemy import select
from dehergne_util import geo_entity_wikidata_id

geo1, geo2, geo3 = tlnb.db.get_model(['geo1','geo2','geo3'])
stmt = select(geo1).where(geo1.inside == 'deh-chre-1701').order_by(geo1.the_line)

place_list = []

with tlnb.db.session() as session:
    result = session.execute(stmt).fetchall()
    for province, in result:
        comment, wikidata = geo_entity_wikidata_id(province, if_missing='No wikidata')
        # print(province.id, province.name, wikidata, comment)
        place_list.append({'province': province.name,
                           'id': province.id,
                           'level': 'province',
                           'name': province.name,
                           'name_original': province.with_extra_info().name,
                           'province_wikidata_id': wikidata,
                           'wikidata_id': wikidata,
                           'the_line': province.the_line,
                           'comment': comment})
        fous = session.execute(select(geo2).where(geo2.inside == province.id)).fetchall()
        province_wikdata = wikidata
        for fou, in fous:

            comment, wikidata = geo_entity_wikidata_id(fou, if_missing='No wikidata')
            # print(' ', fou.name,  wikidata, comment)
            place_list.append({'province': province.name,
                               'province_id': province.id,
                               'province_wikidata_id': province_wikdata,
                               'id': fou.id,
                               'level': 'fou',
                               'fou':fou.name,
                               'name': fou.name,
                               'name_original': fou.with_extra_info().name,
                               'fou_wikidata_id': wikidata,
                               'wikidata_id': wikidata,
                               'the_line': fou.the_line,
                               'comment': comment})
            geo3s = session.execute(select(geo3).where(geo3.inside == fou.id)).fetchall()
            fou_wikidata = wikidata
            for tcheou_hien, in geo3s:
                comment, wikidata = geo_entity_wikidata_id(tcheou_hien, if_missing='No wikidata')
                # print('   ', tcheou_hien.name, wikidata,comment,  )
                place_list.append({'province': province.name,
                                    'province_id': province.id,
                                    'province_wikidata_id': province_wikdata,
                                    'fou_id': fou.id,
                                    'fou_wikidata_id': fou_wikidata,
                                    'id': tcheou_hien.id,
                                    'level': 'tcheou-hien',
                                    'fou':fou.name,
                                    'name': tcheou_hien.name,
                                    'name_original': tcheou_hien.with_extra_info().name,
                                    'wikidata_id': wikidata,
                                    'the_line': tcheou_hien.the_line,
                                    'comment': comment})


#### Create a dataframe with the locations in 1701



In [ ]:
pd.set_option('display.max_rows', 300)
# create a dataframe from the list
places_1701_df = pd.DataFrame(place_list)
places_1701_df['year'] = 1701
cols=['year','id','level','province',  'fou','name', 'name_original', 'province_wikidata_id', 'fou_wikidata_id','wikidata_id', 'comment','the_line']
places_1701_df.info()
# 'the_line' is the order of the places in the list
places_1701_df[cols].sort_values(by=['the_line']).head()

### Export the locations in 1701 

Export list of residences in 1701 to a file and list of wikidata
ids so they can be used to fetch coordinates and other information.

Ensure that notebook `wikidata-linked-data.ipynb` is run after the next cell to update cache of linked data.


In [ ]:
places_1701_df.to_excel('../inferences/residences-1701.xlsx', index=False)
# save to wikidata ids directory
places_1701_df['wikidata_id'].to_csv('../inferences/wikidata-references/residences-1701.csv', index=False)

#### Link with wikidata

Requires that `wikidata-linked-data.ipynb` has been run first to generate the `locations_wikidata_info` file used here.

In [ ]:
# load wikidata data
from dehergne_util import locations_wikidata_info_file

# load the locations wikidata info from xlsx file (again, just in case it was updated
print("Loading locations wikidata info from", locations_wikidata_info_file)
locations_wikidata_info = pd.read_excel(locations_wikidata_info_file, dtype=str)
locations_wikidata_info.set_index('wikidata_id', inplace=True)

# merge the two dataframes
merged_df = pd.merge(places_1701_df, locations_wikidata_info, on='wikidata_id', how='left')
cols = ['level', 'province', 'fou', 'name', 'wikidata_id', 'coordinates','latitude','longitude','comment', 'province_wikidata_id', 'fou_wikidata_id']
merged_df[cols].sort_values(by=['province', 'fou', 'name']).head(10)

#### Scan for coordinates

In [ ]:
import re
from dehergne_util import extract_coordinates

# set the coordinates for the places that have coordinates in the comment column
# interate rows with iterrows
for index, row in merged_df[merged_df['coordinates'].isna()].iterrows():
    # print()
    # print("Looking for coordinates of:", row['id'], row['name'])
    # if row['comment'] is not NaN and contains "coordinates", "latitude", or "longitude"
    if not re.search(r'coordinates:|latitude:|longitude:', row['comment'], flags=re.IGNORECASE):
        # print("No coordinates found in comment for row:", index, row['name'])
        continue
    coords = extract_coordinates(row['comment'])
    if coords:
        print()
        print(row['comment'])
        print("Found coordinates:", coords)
        merged_df.at[index, 'coordinates'] = coords
        merged_df.at[index, 'latitude'] = coords[0]
        merged_df.at[index, 'longitude'] = coords[1]


### List places with no coordinates

These are the ones that where not geo referenced so far.

In [ ]:
# List rows in merged_df where the 'coordinates' column is NaN
missing_coords_df = merged_df[merged_df['coordinates'].isna()]
missing_coords_df

## Map the structure of residences in 1701

In [ ]:
from textwrap import wrap
import plotly.graph_objects as go

# generate a dictionary for quick lookup with id as key
locations_wikidata_dict = {row['id']: row for index, row in merged_df.iterrows()}

# Define marker styles for each level
level_styles = {
    'province': {'color': 'red', 'size': 12},
    'fou': {'color': 'blue', 'size': 8},
    'tcheou-hien': {'color': 'green', 'size': 5}
}

fig = go.Figure()


for level, style in level_styles.items():
    fig.add_trace(go.Scattermap(
        lon=[None],
        lat=[None],
        mode='markers',
        marker=dict(color=style['color'], size=style['size']),
        name=level
    ))
# Add lines for 'fou' and 'tcheou_hien'
for _, row in merged_df.iterrows():
    level = row.get('level')
    if level == 'fou':
        id_origin = row.get('province_id', None)
        id_destination = row.get('id', None)
        lat_origin = locations_wikidata_dict.get(id_origin, {}).get('latitude', None)
        lon_origin = locations_wikidata_dict.get(id_origin, {}).get('longitude', None)
        lat_destination = locations_wikidata_dict.get(id_destination, {}).get('latitude', None)
        lon_destination = locations_wikidata_dict.get(id_destination, {}).get('longitude', None)
        if lat_origin and lon_origin and lat_destination and lon_destination:
            fig.add_trace(go.Scattermap(
                lon=[float(lon_origin), float(lon_destination)],
                lat=[float(lat_origin), float(lat_destination)],
                mode='lines',
                line=dict(width=1, color='red'),
                showlegend=False
            ))
    elif level == 'tcheou-hien':
        id_origin = row.get('fou_id', None)
        id_destination = row.get('id', None)
        lat_origin = locations_wikidata_dict.get(id_origin, {}).get('latitude', None)
        lon_origin = locations_wikidata_dict.get(id_origin, {}).get('longitude', None)
        lat_destination = locations_wikidata_dict.get(id_destination, {}).get('latitude', None)
        lon_destination = locations_wikidata_dict.get(id_destination, {}).get('longitude', None)

        if lat_origin and lon_origin and lat_destination and lon_destination:
            fig.add_trace(go.Scattermap(
                lon=[float(lon_origin), float(lon_destination)],
                lat=[float(lat_origin), float(lat_destination)],
                mode='lines',
                line=dict(width=1, color='blue'),
                showlegend=False
            ))
# Add Markers for each location
for index, row in merged_df.iterrows():
    lat = row.get('latitude')
    lon = row.get('longitude')
    level = row.get('level')
    name = row.get('name')
    english_description = row.get('english_description', '')
    chinese_description = row.get('chinese_description', '')
    wikidata_id = row.get('wikidata_id', '')
    coordinates = row.get('coordinates', '')
    if name == "Camarri":
        print(f"Debugging: {name} with wikidata_id {wikidata_id}")

    if wikidata_id != 'No wikidata':
        english_label = row.get('english_label', '')
        chinese_label = row.get('chinese_label', '')
        portuguese_label = row.get('portuguese_label', '')
        english_description = row.get('english_description', '')
        chinese_description = row.get('chinese_description', '')
        portuguese_description = row.get('portuguese_description', '')
        wikidata_label = (
                         f"en:{english_label} ({english_description})<br>"
                         f"zh:{chinese_label} ({chinese_description})<br>"
                         f"pt:{portuguese_label} ({portuguese_description})"
        )
    else:
        wikidata_label = "No wikidata"
        english_label = ''
        chinese_label = ''
        portuguese_label = ''
        english_description = ''
        chinese_description = ''
        portuguese_description = ''

    name_original = row.get('name_original', '')
    comment = "<br>".join(wrap(name_original, width=40))
    hover_text = f"name: {name}<br>original: {comment}<br><br>wikidata ({wikidata_id})<br>{wikidata_label} <br>coordinates: {coordinates}"


    if level == 'tcheou-hien':
        pass
    if pd.notnull(lat) and pd.notnull(lon) and level in level_styles:
        style = level_styles[level]
        fig.add_trace(go.Scattermap(
            lon=[float(lon)],
            lat=[float(lat)],
            mode='markers+text',
            marker=dict(color=style['color'], size=style['size']),
            text=[name],
            textposition='top center',
            hovertext=[hover_text],
            hoverinfo='text',
            hoverlabel=dict(bgcolor='white', font_size=12),
            showlegend=False
        ))

# fig.update_layout(mapbox_style="open-street-map")
# fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.update_traces(textfont=dict(size=6))
fig.update_layout({
    "map": {"center": {"lat": 28.0,
                       "lon": 120.0},
            "zoom": 4,
            "style":  "carto-positron",
            }
    },
    title= "Residences 1701",
    annotations=[dict(
        x=0,
        y=-0.1,
        xref="paper",
        yref="paper",
        text="<a href='https://archive.org/details/bhsi37/page/n392/mode/1up'>Source</a>",
        showarrow=False,
        font=dict(size=10)
    )],
    autosize=True,
    )

display(HTML(fig.to_html(include_plotlyjs=True, config={"responsive": True})))
fig.write_html("../inferences/residences_1701_map.html",
               config={"responsive": True},
               include_plotlyjs=True,
               auto_open=True)


## Append the two lists 1644 and 1701


In [ ]:
# append places_1644_df and places_1701_df into a single dataframe
places_1644_1701_df = pd.concat([places_1644_df, places_1701_df], ignore_index=True)
# export to excel
places_1644_1701_df.to_excel('../inferences/residences-1644-1701.xlsx', index=False)
# export to csv wikidata ids directory
places_1644_1701_df['wikidata_id'].to_csv('../inferences/wikidata-references/residences-1644-1701.csv', index=False)

In [ ]:
places_1644_1701_df.head()

## Show one location

In [ ]:
entity_id = 'deh-r1644-chekiang'
with tlnb.db.session() as session:
    # save the dataframe to the database
    loc = tlnb.db.get_entity(entity_id, session)
    if loc is not None:
        print(loc.to_kleio())

## Listing people in residences

In [ ]:
import re
from collections import namedtuple                                                                                                          
import pandas as pd
from sqlalchemy import select

from timelink.kleio.utilities import convert_timelink_date, format_timelink_date
from timelink.api.models import Geoentity

nattributes = tlnb.db.get_view('nattributes')

def show_nattributes_for_wikidata(wikidata, ident='', session=None):
    """ Show the nattribute for a given wikidata id
    """
    wikidata_uri = 'https://www.wikidata.org/wiki/' + wikidata
    stmt = select(nattributes).where(nattributes.c.the_value == wikidata_uri).order_by(nattributes.c.the_date)
    results = session.execute(stmt).fetchall()
    for result in results:
        fdate = format_timelink_date(result.the_date)
        print(f"{ident}{fdate} {result.name} ({result.the_type}) {result.id} {result.eobs} ({result.the_date})")

def get_nattributes_for_wikidata(wikidata, session=None):                                                                                       
      """Get nattributes for a given wikidata id as a list of named tuples.                                                                       

        1. Returns data instead of printing - Returns a list of NattributeRecord named tuples                                                           
        2. Same query logic - Uses identical SQL query and data processing                                                                              
        3. Structured output - Each record has named fields for easy access:                                                                            
            - the_date: Formatted date string                                                                                                             
            - name: Attribute name                                                                                                                        
            - the_type: Attribute type                                                                                                                    
            - id: Entity ID                                                                                                                               
            - eobs: Entity observation                                                                                                                    
            - raw_date: Original date value                                                                                                               
                                                                                                                                                                                                                                                                              
        Usage example:                                                                                                                                  
        # Get records as list                                                                                                                           
        records = get_nattributes_for_wikidata('Q16967', session=session)                                                                               
                                                                                                                                                        
        # Access individual fields                                                                                                                      
        for record in records:                                                                                                                          
            print(f"{record.the_date} {record.name} ({record.the_type})")                                                                               
                                                                                                                                                        
        # Convert to DataFrame if needed                                                                                                                
        df = pd.DataFrame(records)                                                                                                                      
                                                                                                                                                        
        This gives you the same data but in a programmatic format that you can manipulate, filter, or analyze further.                                                                                                                                        
            Args:                                                                                                                                       
                wikidata: Wikidata ID (e.g., 'Q16967')                                                                                                  
                session: SQLAlchemy database session                                                                                                    
                                                                                                                                                  
      Returns:                                                                                                                                    
          List of named tuples with fields:                                                                                                       
          - the_date: formatted date string                                                                                                       
          - name: attribute name                                                                                                                  
          - the_type: attribute type                                                                                                              
          - id: entity ID                                                                                                                         
          - eobs: entity observation                                                                                                              
          - raw_date: original date value          

      """                                                                                                                                                                                                                                                                            
      NattributeRecord = namedtuple('NattributeRecord', ['the_date', 'the_class', 'name', 'the_type', 'id', 'eobs', 'raw_date'])                               
      wikidata_uri = 'https://www.wikidata.org/wiki/' + wikidata                                                                                  
      stmt = select(nattributes).where(nattributes.c.the_value == wikidata_uri).order_by(nattributes.c.the_date)                                  
      results = session.execute(stmt).fetchall()                                                                                                  
                                                                                                                                                  
      records = []                                                                                                                                
      for result in results:                                                                                                                      
          fdate = format_timelink_date(result.the_date)                                                                                           
          record = NattributeRecord(                                                                                                              
              the_date=fdate,                                                                                                                     
              the_class=result.pom_class,                                                                                                                 
              name=result.name,                                                                                                                   
              the_type=result.the_type,                                                                                                           
              id=result.id,                                                                                                                       
              eobs=result.eobs,                                                                                                                   
              raw_date=result.the_date                                                                                                            
          )                     
          if wikidata == 'Q16967' and result.pom_class == 'geoentity':
                pass                                                                                                                    
          records.append(record)                                                                                                                  
                                                                                                                                                  
      return records      

In [ ]:
nattributes = tlnb.db.get_view('nattributes')
tlnb.db.describe(nattributes)

In [ ]:
geo1, geo2, geo3 = tlnb.db.get_model(['geo1', 'geo2', 'geo3'])
stmt = select(geo1).order_by(geo1.the_source, geo1.the_line).where(geo1.inside.in_(['deh-chre-1644','deh-chre-1701']))

place_list = []
name_list = []
with tlnb.db.session() as session:
    result = session.execute(stmt).fetchall()
    for province, in result:
        print()
        comment, wikidata = geo_entity_wikidata_id(province, if_missing='No wikidata')
        print(province.name, wikidata, comment)
        province_record = {'id': province.id, 'level': 'province', 'province': province.name,'fou':'',  'tcheou_hien': '', 'wikidata': wikidata, 'comment': comment}
        show_nattributes_for_wikidata(wikidata, '  ', session=session)
        entities = get_nattributes_for_wikidata(wikidata, session)
        for entity in entities:
            # merge entity record with province record and add to place_list
            record = province_record.copy()
            record.update(entity._asdict())
            if entity.the_class == 'person':
                name_list.append(record)
            elif entity.the_class == 'geoentity' and record not in name_list:
                place_list.append(record)
        # search all the entities that have this value in one of the attributes
        fous = session.execute(select(geo2).where(geo2.inside == province.id)).fetchall()
        for fou, in fous:
            print()
            comment, wikidata = geo_entity_wikidata_id(fou, if_missing='No wikidata')
            print(' ', fou.name,  wikidata, comment)
            fou_record = {'id': fou.id, 'level': 'fou', 'province': province.name, 'fou':fou.name,  'tcheou_hien': '', 'wikidata': wikidata, 'comment': comment}
            show_nattributes_for_wikidata(wikidata, '   ', session=session)
            entities = get_nattributes_for_wikidata(wikidata, session)
            for entity in entities:
                # merge entity record with fou record and add to place_list
                record = fou_record.copy()
                record.update(entity._asdict())
                if entity.the_class == 'person':
                    name_list.append(record)
                elif entity.the_class == 'geoentity' and record not in name_list:
                    place_list.append(record)
            geo3s = session.execute(select(geo3).where(geo3.inside == fou.id)).fetchall()
            for tcheou_hien, in geo3s:
                print()
                comment, wikidata = geo_entity_wikidata_id(tcheou_hien, if_missing='No wikidata')
                print('   ', tcheou_hien.name, wikidata,comment,  )
                tcheou_record = {'id': tcheou_hien.id, 'level': 'tcheou-hien', 'province': province.name, 'fou':fou.name,  'tcheou_hien': tcheou_hien.name, 'wikidata': wikidata, 'comment': comment}
                show_nattributes_for_wikidata(wikidata, '     ', session=session)
                entities = get_nattributes_for_wikidata(wikidata, session)
                for entity in entities:
                    # merge entity record with tcheou_hien record and add to place_list
                    record = tcheou_record.copy()
                    record.update(entity._asdict())
                    if entity.the_class == 'person':
                        name_list.append(record)
                    elif entity.the_class == 'geoentity' and record not in name_list:
                        place_list.append(record)
                    


### Create a dataframe with the people in residences

Loop through the list of residences in 1644 and 1701 and create a dataframe with the people in each residence.

Note that we are merging two lists of residences: 1644 and 1701, so we will have in some cases two entries for the same wikidata id (one for each residence). Both entries will share the same wikidata id of the location but will have different dates (`the_date`) and different ids (`id`). Each row will also preserve the specific comment of the source of the corresponding year

Example:

| id | level | province | fou | tcheou_hien | wikidata | comment | the_date | the_class | name | the_type | eobs | raw_date |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| deh-r1644-chekiang | province | Chekiang | | | Q16967 | Tche-kiang, today Zhejiang, 浙江, @dehergne:396 | 1644 | geoentity | Chekiang | geoentity:name@wikidata | | 16440000 |
| deh-r1701-chekiang | province | Chekiang | | | Q16967 | Tche-kiang, today Zhejiang, 浙江, @dehergne:396 | 1701 | geoentity | Chekiang | geoentity:name@wikidata | | 1701000 |

Example:

| id | level | province | fou | tcheou_hien | wikidata | comment | the_date | the_class | name | the_type | eobs | raw_date |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| deh-r1644-chekiang | province | Chekiang | | | Q16967 | Tche-kiang, today Zhejiang, 浙江, @dehergne:396 | 1644 | geoentity | Chekiang | geoentity:name@wikidata | | 16440000 |
| deh-r1701-chekiang | province | Chekiang | | | Q16967 | Tche-kiang, today Zhejiang, 浙江, @dehergne:396 | 1701 | geoentity | Chekiang | geoentity:name@wikidata | | 1701000 |

In [ ]:
# create a Dataframe with the place_list, remove duplicates
# duplicates are generated when the same wikidata id appears in more than one record of places
places_df = pd.DataFrame(place_list)
places_df = places_df.drop_duplicates(subset=['id'])
places_df.info()
places_df.head(10)


In [ ]:
# Create a DataFrame from the place_list
names_df = pd.DataFrame(name_list)
names_df.info()
names_df.sort_values(["province","fou","tcheou_hien","raw_date","the_class",]).head(10)

### Export the list of people per residence to Excel

In [ ]:
residences_df = pd.concat([places_df, names_df], ignore_index=True)
sort_by = ['province', 'fou', 'tcheou_hien', 'the_class', 'wikidata','the_date']
residences_df = residences_df.sort_values(by=sort_by)
residences_df.head(10)

## Export list of residences and people to excel

In [ ]:
residences_df.to_excel('../inferences/residences-with_people.xlsx', index=False)
